In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym
import numpy as np
import random
from collections import deque
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256, log_std_min=-20, log_std_max=2):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        mean = self.mean(x)
        log_std = self.log_std(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mean, log_std

    def sample(self, state):
        mean, log_std = self.forward(state)
        std = log_std.exp()

        
        normal = torch.distributions.Normal(0, 1)
        z = normal.sample().to(device)
        action = mean + std * z
        log_prob = normal.log_prob(z) - torch.log(1 - torch.tanh(action).pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=1, keepdim=True)
        log_prob -= (2 * (np.log(2) - action - F.softplus(-2 * action))).sum(dim=1, keepdim=True)

        
        action = torch.tanh(action)

        return action, log_prob, mean, std


class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        samples = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*samples)
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards),
            np.array(next_states),
            np.array(dones)
        )

    def __len__(self):
        return len(self.buffer)


env = gym.make('Pendulum-v1')


state_dim = env.observation_space.shape[0]  
action_dim = env.action_space.shape[0]      
hidden_dim = 256
buffer_size = 100_000
batch_size = 256
lr = 3e-4
gamma = 0.99
tau = 0.005  
alpha = 0.2  
update_every = 50
num_episodes = 200
max_steps = 200


actor = PolicyNetwork(state_dim, action_dim, hidden_dim).to(device)
q1 = QNetwork(state_dim, action_dim, hidden_dim).to(device)
q2 = QNetwork(state_dim, action_dim, hidden_dim).to(device)

q1_target = QNetwork(state_dim, action_dim, hidden_dim).to(device)
q2_target = QNetwork(state_dim, action_dim, hidden_dim).to(device)
q1_target.load_state_dict(q1.state_dict())
q2_target.load_state_dict(q2.state_dict())


actor_optimizer = optim.Adam(actor.parameters(), lr=lr)
q1_optimizer = optim.Adam(q1.parameters(), lr=lr)
q2_optimizer = optim.Adam(q2.parameters(), lr=lr)


replay_buffer = ReplayBuffer(buffer_size)


def soft_update(q_target, q, tau):
    for target_param, param in zip(q_target.parameters(), q.parameters()):
        target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)


for episode in range(num_episodes):
    state = env.reset()[0]
    total_reward = 0

    for step in range(max_steps):
        
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            if len(replay_buffer) < 1000:
                action = env.action_space.sample()
            else:
                mean, log_std = actor.forward(state_tensor)
                std = log_std.exp()
                action_sample = torch.distributions.Normal(mean, std).sample()
                action_raw = torch.tanh(action_sample)
                action = action_raw.cpu().numpy()[0]

        
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

        
        replay_buffer.push(state, action, reward, next_state, done)
        state = next_state

        
        if len(replay_buffer) >= batch_size and step % update_every == 0:
            states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
            states = torch.FloatTensor(states).to(device)
            actions = torch.FloatTensor(actions).to(device)
            rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
            next_states = torch.FloatTensor(next_states).to(device)
            dones = torch.FloatTensor(dones).unsqueeze(1).to(device)

            
            with torch.no_grad():
                next_actions, next_log_probs, _, _ = actor.sample(next_states)
                q1_next = q1_target(next_states, next_actions)
                q2_next = q2_target(next_states, next_actions)
                min_q_next = torch.min(q1_next, q2_next) - alpha * next_log_probs
                q_target = rewards + gamma * (1 - dones) * min_q_next

            q1_loss = F.mse_loss(q1(states, actions), q_target)
            q2_loss = F.mse_loss(q2(states, actions), q_target)

            q1_optimizer.zero_grad()
            q1_loss.backward()
            q1_optimizer.step()

            q2_optimizer.zero_grad()
            q2_loss.backward()
            q2_optimizer.step()

            
            new_actions, log_probs, _, _ = actor.sample(states)
            q1_new = q1(states, new_actions)
            q2_new = q2(states, new_actions)
            min_q_new = torch.min(q1_new, q2_new)

            actor_loss = (alpha * log_probs - min_q_new).mean()

            actor_optimizer.zero_grad()
            actor_loss.backward()
            actor_optimizer.step()

            
            soft_update(q1_target, q1, tau)
            soft_update(q2_target, q2, tau)

        if done:
            break

    if episode % 20 == 0:
        print(f"Episode {episode}, Total Reward: {total_reward:.2f}")

print("Training finished.")